# DCF from statement UFCF

Project **unlevered free cash flow** in the model, then call `evaluate_dcf` with **WACC** and a **terminal value** specification. Bridge **enterprise value to equity** with net debt or an equity bridge.

## Concept

`evaluate_dcf` discounts **forecast-period** UFCF, adds **PV of terminal value** (Gordon growth or exit multiple), and subtracts **net debt** (from `total_debt` / `cash` at the valuation boundary) unless you pass `net_debt_override`. Metadata `meta.currency` must be an ISO code (we patch JSON after `build()`).

In [ ]:
from finstack.statements import ModelBuilder, Evaluator, FinancialModelSpec
from finstack.statements_analytics import run_sensitivity, evaluate_scenario_set, goal_seek, trace_dependencies, explain_formula, run_variance, evaluate_dcf, run_corporate_analysis, pl_summary_report, credit_assessment_report
import json
print("Loaded finstack.statements + finstack.statements_analytics")

from finstack.statements_analytics import evaluate_dcf

PERIODS = ["2026", "2027", "2028", "2029", "2030"]


def s(v):
    return list(zip(PERIODS, v))


b = ModelBuilder("dcf-demo")
b.periods("2026..2030", "2025")  # all forecast for UFCF stream
b.value("ufcf", s([18.0, 19.5, 21.0, 22.0, 23.0]))
b.value("total_debt", s([80.0, 78.0, 75.0, 72.0, 70.0]))
b.value("cash", s([12.0, 13.0, 14.0, 15.0, 16.0]))
spec = b.build()
raw = json.loads(spec.to_json())
raw.setdefault("meta", {})["currency"] = "USD"
spec_usd = FinancialModelSpec.from_json(json.dumps(raw))
model_json = spec_usd.to_json()

cost_equity = 0.12
cost_debt_pretax = 0.06
tax_rate = 0.25
debt_weight = 0.35
equity_weight = 0.65
wacc_manual = equity_weight * cost_equity + debt_weight * cost_debt_pretax * (1.0 - tax_rate)
wacc = wacc_manual  # ~0.09375 (often rounded to 0.0938); DCF uses this exact build
print("Illustrative WACC build:", round(wacc_manual, 4), "(used in evaluate_dcf below)")

tv_gordon = json.dumps({"type": "gordon_growth", "growth_rate": 0.025})
tv_exit = json.dumps({"type": "exit_multiple", "terminal_metric": 24.0, "multiple": 9.0})

try:
    out_g = evaluate_dcf(model_json, wacc, tv_gordon, ufcf_node="ufcf")
    print("Gordon terminal — equity_value:", out_g.get("equity_value"), "EV:", out_g.get("enterprise_value"))
except Exception as e:
    print("evaluate_dcf (Gordon) unavailable or failed:", e)

try:
    out_x = evaluate_dcf(model_json, wacc, tv_exit, ufcf_node="ufcf")
    print("Exit multiple terminal — equity_value:", out_x.get("equity_value"))
except Exception as e:
    print("evaluate_dcf (exit multiple) unavailable or failed:", e)

bridge = json.dumps(
    {
        "total_debt": 70.0,
        "cash": 16.0,
        "preferred_equity": 0.0,
        "minority_interest": 0.0,
        "non_operating_assets": 2.0,
        "other_adjustments": [["pension", 1.0]],
    }
)
try:
    out_br = evaluate_dcf(
        model_json,
        wacc,
        tv_gordon,
        ufcf_node="ufcf",
        equity_bridge_json=bridge,
    )
    print("With equity_bridge_json — equity_value:", out_br.get("equity_value"), "net_debt field:", out_br.get("net_debt"))
except Exception as e:
    print("evaluate_dcf with bridge failed:", e)


In [ ]:
print("DCF sensitivity grid")
# Sensitivity grid: WACC vs terminal growth (Gordon)

growth_rates = [0.015, 0.025, 0.035]
waccs = [0.085, 0.095, 0.105]
print("WACC \ g -> equity_value (Gordon)")
for w in waccs:
    row = []
    for g in growth_rates:
        if g >= w:
            row.append(None)
            continue
        tv = json.dumps({"type": "gordon_growth", "growth_rate": g})
        try:
            o = evaluate_dcf(model_json, w, tv, ufcf_node="ufcf")
            row.append(round(float(o.get("equity_value", 0.0)), 2))
        except Exception:
            row.append(None)
    print(w, row)
print("Done.")


## Takeaways

- DCF needs **forecast UFCF** rows, **`meta.currency`**, and **balance-sheet anchors** (or overrides) for net debt.
- **Terminal value** tagging follows `TerminalValueSpec` JSON: `gordon_growth` vs `exit_multiple`.
- A full **WACC** build belongs in your assumptions; the grid shows how sensitive equity value is to **rate + terminal growth**.